In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from scipy import stats
import warnings
import sys
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import swifter
import matplotlib.dates as mdates
palette = "Set1"
sys.path.append('../')
pd.options.mode.chained_assignment = None 



In [2]:
#combine all CSV's of the same data
files = open("bees/files-short.txt")
filenames = []
while True:
    line = files.readline().strip()
    if not line:
        break
    filenames.append("bees/" + line)

vdf = pd.concat(map(pd.read_csv, filenames), ignore_index=True)
vdf['track_starttime'] = pd.to_datetime(vdf['track_starttime'])
vdf['track_endtime'] = pd.to_datetime(vdf['track_endtime'])
df = vdf.copy()

In [ ]:
def groupbyseconds(vdf,sec):
    tracks = (vdf.groupby(['track_id','track_tagid'], as_index=False)
            .agg(start=('track_starttime','min'),
                 end=('track_endtime','max')))


    #sort by video name, track id and start
    tracks = tracks.sort_values(['track_tagid','start'])
    
    
    #obtain breakpoints of 5 second gaps
    tracks['gap_s'] = (
        tracks['start'] - tracks.groupby(['track_tagid'])['end'].shift(1)
    ).dt.total_seconds()

    #mark as breakpoints
    tracks['split'] = tracks['gap_s'].isna() | (tracks['gap_s'] > sec) | (tracks['gap_s'] < 0)
    
    #segment to obtain track values by individual ids
    tracks['segment'] = tracks.groupby(['track_tagid'])['split'].cumsum()
    
    #group by unique ids and tracks
    tracks['group_id'] = tracks.groupby(['track_tagid', 'segment']).ngroup()

    group_members = (
        tracks.groupby(['track_tagid', 'group_id'])['track_id']
        .apply(lambda s: tuple(sorted(s)))
        .reset_index(name='members')
    )

    # stable group_id: tagid + exact member list
    group_members['global_id'] = group_members.apply(
        lambda row: f"{row['track_tagid']}__{'_'.join(map(str, row['members']))}",
        axis=1
    )

    tracks = tracks.merge(
        group_members[['track_tagid', 'group_id', 'global_id']],
        on=['track_tagid', 'group_id'],
        how='left'
    )
    
    vdf = vdf.merge(
        tracks[['track_id', 'track_tagid','global_id','group_id']],
        on=['track_id','track_tagid'],
        how='left'
    )
    vdf = vdf.dropna(subset=['group_id'])

    return vdf

In [4]:
def groupbyseconds(vdf,sec):
    tracks = (vdf.groupby(['track_id','track_tagid'], as_index=False)
            .agg(start=('track_starttime','min'),
                 end=('track_endtime','max')))


    #sort by video name, track id and start
    tracks = tracks.sort_values(['track_tagid','start'])
    
    
    #obtain breakpoints of 5 second gaps
    tracks['gap_s'] = (
        tracks['start'] - tracks.groupby(['track_tagid'])['end'].shift(1)
    ).dt.total_seconds()

    #drop overlaps
    overlap_current = tracks['gap_s'] < 0
    overlap_previous = overlap_current.groupby(tracks['track_tagid']).shift(-1, fill_value=False)
    tracks = tracks[~(overlap_current | overlap_previous)].copy()
    
    #mark as breakpoints
    tracks['split'] = tracks['gap_s'].isna() | (tracks['gap_s'] > sec)
    
    #segment to obtain track values by individual ids
    tracks['segment'] = tracks.groupby(['track_tagid'])['split'].cumsum()
    
    #group by unique ids and tracks
    tracks['group_id'] = tracks.groupby(['track_tagid', 'segment']).ngroup()

    group_members = (
        tracks.groupby(['track_tagid', 'group_id'])['track_id']
        .apply(lambda s: tuple(sorted(s)))
        .reset_index(name='members')
    )

    # stable group_id: tagid + exact member list
    group_members['global_id'] = group_members.apply(
        lambda row: f"{row['track_tagid']}__{'_'.join(map(str, row['members']))}",
        axis=1
    )

    tracks = tracks.merge(
        group_members[['track_tagid', 'group_id', 'global_id']],
        on=['track_tagid', 'group_id'],
        how='left'
    )
    
    vdf = vdf.merge(
        tracks[['track_id', 'track_tagid','global_id','group_id']],
        on=['track_id','track_tagid'],
        how='left'
    )
    vdf = vdf.dropna(subset=['group_id'])

    return vdf

In [12]:
def graphplots(vdf,seconds):
    for i in vdf['group_id'].unique():
        detections = vdf[vdf['group_id'] == i]
    
        positions = detections['cy']
        x = detections['frame']
        if(len(x) != len(x.unique())):
            continue
        plot = sns.scatterplot(x=x,y=positions,hue=detections['track_id'],palette=palette)
        plot.set_ylim(-200, 1600)
        plot.axhline(y=0,color="black",linestyle ="--")
        plot.axhline(y=1400,color="black",linestyle ="--")
        plot.set_title(f"event={i} merge={seconds}s tag={detections['track_tagid'].iloc[0]}")
        plt = plot.get_figure()
        plt.savefig(f"{seconds}second/group_{int(i):04}_track_{detections['track_id'].iloc[0]:05}_{seconds}s.png")
        plt.clf()


In [10]:
def graph_distance(vdf, seconds):
    tracks = (vdf.groupby(['track_id','track_tagid'], as_index=False)
            .agg(start=('track_starttime','min'),
                 end=('track_endtime','max'),
                firstframe=('frame','min'),
                lastframe=('frame','max')))
    tracks = tracks.sort_values(['track_tagid','start'])
    tracks['dif'] = (tracks['start'] - tracks['end'].shift(1)).dt.total_seconds()
    for index, row in tracks.iterrows():
        thetracks = tracks[tracks['track_tagid'] == row['track_tagid']].reset_index(drop=True)
        tracknum = len(thetracks)
        if tracknum < 2:
            continue
    
        i = thetracks.index[
            (thetracks['track_id'] == row['track_id'])
        ][0]
        neighbors = get_neighbors(thetracks,i)
        neighbors = neighbors[neighbors['dif'] < seconds]
        #print(neighbors)
    
        neighbors = pd.concat([neighbors,row])
        
    
        neighbor_keys = neighbors[['track_id', 'track_tagid']].drop_duplicates()

        data = vdf.merge(
            neighbor_keys,
            on=['track_id', 'track_tagid'],
            how='inner'
        ).copy()
        if len(data['track_id'].unique()) < 2:
            continue
        #print(data)
        
        positions = data['cy']
        data['timestamp'] = pd.to_datetime(data['timestamp'])
        x = data['timestamp']  
        if(len(x) != len(x.unique())):
            continue
        plot = sns.scatterplot(x=x,y=positions,hue=data['track_id'],palette=palette)
        plot.set_ylim(-200, 1600)
        #plot.set_xlim(row['start'] - pd.Timedelta(seconds = 5), row['end'] + pd.Timedelta(seconds = 5))
        xleft = pd.to_datetime(row['start']) - pd.Timedelta(seconds=seconds)
        xright = pd.to_datetime(row['end']) + pd.Timedelta(seconds=seconds)  
        xmin = x.min()
        xmax = x.max()
    
        ticks = pd.to_datetime(np.linspace(xleft.value, xright.value, 10))
        plot.set_xlim(xleft, xright)
        plot.set_xticks(ticks) 
        plot.xaxis.set_major_formatter(mdates.DateFormatter('%M:%S'))
        plot.axvline(x=pd.to_datetime(row['start']), color='black', linestyle=':')
        plot.axvline(x=pd.to_datetime(row['end']), color='black', linestyle=':')
        plot.text(x=pd.to_datetime(row['end']),y=0.5,s=f"{seconds}s+")
        plot.text(x=pd.to_datetime(row['start']),y=0.5,s=f"{seconds}s-",ha='right')
        plot.set_title(f"tracks={data['track_id'].unique()}, tag={row['track_tagid']}")
        plot = plot.get_figure()       
        plt.xticks(ticks, rotation=45)
        plot.show()
        plot.savefig(f"distance/{seconds}second/track{int(row['track_id']):05}.png")
        plt.clf()
    

In [7]:
def makecsv(vdf,seconds):
    with open(f"{seconds}seconds.csv", "w") as f:
        f.write("group_id,event,ambiguousflag,track_id,filename,video,datetime,tagid,mergethreshold,eventid\n")
        for group_id in np.sort(vdf['group_id'].unique()):
            group = vdf[vdf['group_id'] == group_id].sort_values(by=['track_starttime'])
            ambiguous = "x" if len(group['frame'].unique()) != len(group['frame']) else ""
            filename = f"{seconds}second/track_{group['track_id'].iloc[0]:05}_{seconds}s.png"
            f.write(f"{group_id},,{ambiguous},{group['track_id'].iloc[0]:05},{filename},{group['video_name'].iloc[0]},{group['timestamp'].iloc[-1]},{group['track_tagid'].iloc[0]},{seconds}s,{group['global_id'].iloc[0]}\n")
            
            

In [20]:
def get_neighbors(df, i):
    return df.iloc[max(i-1, 0): min(i+2, len(df))]

### GRAPH GENERATION

## 5 second distance graphs

In [21]:
graph_distance(vdf,5)

C:\Users\kashi\AppData\Local\Temp\ipykernel_2696\3596507421.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plot.show()
C:\Users\kashi\AppData\Local\Temp\ipykernel_2696\3596507421.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plot.show()
C:\Users\kashi\AppData\Local\Temp\ipykernel_2696\3596507421.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plot.show()
C:\Users\kashi\AppData\Local\Temp\ipykernel_2696\3596507421.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plot.show()
C:\Users\kashi\AppData\Local\Temp\ipykernel_2696\3596507421.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plot.show()
C:\Users\kashi\AppData\Local\Temp\ipykernel_2696\3596507421.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plot.show()
C:\Users\kashi\AppData\Local\Temp\ipykernel_2696\3596507421.py:60: Use

<Figure size 640x480 with 0 Axes>

## 5 second split graphs

In [8]:
vdf = df.copy()
vdf = groupbyseconds(vdf,5)
vdf5 = vdf.copy()
#graphplots(vdf,5)

 

#vdf.to_csv("5secondActivity.csv")

In [10]:
vdf = df.copy()
vdf = groupbyseconds(vdf,5)

In [13]:
graphplots(vdf,5)



<Figure size 640x480 with 0 Axes>

In [14]:
#generate empty csv to classify
#makecsv(vdf,5)

In [15]:
#fill with previously classified events

cheatsheet5 = pd.read_csv("5seconds.csv")
cheatsheetfull = pd.read_csv("5secondsfull.csv")

In [16]:
#fill same events with existing classification

#cheatsheet20 = pd.read_csv("20seconds.csv")

mapping = (
    cheatsheetfull[['eventid', 'event']]
    .drop_duplicates(subset='eventid')
    .set_index('eventid')['event']
)

cheatsheet5['event'] = cheatsheet5['eventid'].map(mapping)


mapping = (
    cheatsheetfull[['eventid', 'ambiguousflag']]
    .drop_duplicates(subset='eventid')
    .set_index('eventid')['ambiguousflag']
)

cheatsheet5['ambiguousflag'] = cheatsheet5['eventid'].map(mapping)


#cheatsheet20.to_csv("")
cheatsheet5.to_csv("5seconds.csv",index=False)

## 20 seconds

In [17]:
vdf = df.copy()
#graph_distance(vdf,20)
vdf = groupbyseconds(vdf,20)
graphplots(vdf,20)
makecsv(vdf,20)
#vdf.to_csv("20secondActivity.csv")

<Figure size 640x480 with 0 Axes>

In [8]:
cheatsheet5 = pd.read_csv("5seconds.csv")

In [18]:
#fill same events with existing classification

cheatsheet20 = pd.read_csv("20seconds.csv")

mapping = (
    cheatsheet5[['eventid', 'event']]
    .drop_duplicates(subset='eventid')
    .set_index('eventid')['event']
)

cheatsheet20['event'] = cheatsheet20['eventid'].map(mapping)


mapping = (
    cheatsheet5[['eventid', 'ambiguousflag']]
    .drop_duplicates(subset='eventid')
    .set_index('eventid')['ambiguousflag']
)

cheatsheet20['ambiguousflag'] = cheatsheet20['eventid'].map(mapping)


#cheatsheet20.to_csv("")
cheatsheet20.iloc[1]
cheatsheet20.to_csv("20seconds.csv",index=False)

## all tracks

In [24]:
vdf['unique_id'] = vdf.groupby(["track_id"]).ngroup()

len(vdf['unique_id'].unique())

1690

## Merging <0s differences

In [18]:
def groupbyseconds(vdf,sec):
    tracks = (vdf.groupby(['track_id','track_tagid'], as_index=False)
            .agg(start=('track_starttime','min'),
                 end=('track_endtime','max')))


    #sort by video name, track id and start
    tracks = tracks.sort_values(['track_tagid','start'])
    
    
    #obtain breakpoints of 5 second gaps
    tracks['gap_s'] = (
        tracks['start'] - tracks.groupby(['track_tagid'])['end'].shift(1)
    ).dt.total_seconds()
    
    #mark as breakpoints
    tracks['split'] = tracks['gap_s'].isna() | (tracks['gap_s'] > sec)
    
    #segment to obtain track values by individual ids
    tracks['segment'] = tracks.groupby(['track_tagid'])['split'].cumsum()
    
    #group by unique ids and tracks
    tracks['group_id'] = tracks.groupby(['track_tagid', 'segment']).ngroup()

    group_members = (
        tracks.groupby(['track_tagid', 'group_id'])['track_id']
        .apply(lambda s: tuple(sorted(s)))
        .reset_index(name='members')
    )

    # stable group_id: tagid + exact member list
    group_members['global_id'] = group_members.apply(
        lambda row: f"{row['track_tagid']}__{'_'.join(map(str, row['members']))}",
        axis=1
    )

    tracks = tracks.merge(
        group_members[['track_tagid', 'group_id', 'global_id']],
        on=['track_tagid', 'group_id'],
        how='left'
    )
    
    vdf = vdf.merge(
        tracks[['track_id', 'track_tagid','global_id','group_id']],
        on=['track_id','track_tagid'],
        how='left'
    )
    vdf = vdf.dropna(subset=['group_id'])

    return vdf


def graphplots(vdf,seconds):
    for i in vdf['group_id'].unique():
        detections = vdf[vdf['group_id'] == i]
    
        positions = detections['cy']
        x = detections['frame']
        #if(len(x) != len(x.unique())):
        #    continue
        plot = sns.scatterplot(x=x,y=positions,hue=detections['track_id'],palette=palette)
        plot.set_ylim(-200, 1600)
        plot.axhline(y=0,color="black",linestyle ="--")
        plot.axhline(y=1400,color="black",linestyle ="--")
        plot.set_title(f"event={i} merge={seconds}s tag={detections['track_tagid'].iloc[0]}")
        plt = plot.get_figure()
        plt.savefig(f"{seconds}second-merge-sub0/group_{i:04}_track_{detections['track_id'].iloc[0]:05}_{seconds}s.png")
        plt.clf()
